In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm
from pathlib import Path
import random
import matplotlib.pyplot as plt
import json
 
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
DATA_FILENAME = "books_big.parquet"
candidates = list(Path("/kaggle/input").rglob(DATA_FILENAME))
if not candidates:
    raise FileNotFoundError(f"{DATA_FILENAME} not found")
DATA_PATH = candidates[0]
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
 
print(f"Device: {DEVICE}")
df = pd.read_parquet(DATA_PATH)
print(f"Rows: {len(df):,} | Users: {df['user_id'].nunique():,} | Items: {df['item_id'].nunique():,}")
 
# Mappings
user2idx = {uid: i for i, uid in enumerate(df["user_id"].unique())}
item2idx = {iid: i for i, iid in enumerate(df["item_id"].unique())}
idx2item = {v: k for k, v in item2idx.items()}
n_users = len(user2idx)
n_items = len(item2idx)
df["user_idx"] = df["user_id"].map(user2idx)
df["item_idx"] = df["item_id"].map(item2idx)
 
# Leave-one-out split
df = df.sort_values(["user_id", "timestamp"]).copy()
df["_rank"] = df.groupby("user_id")["timestamp"].rank(method="first", ascending=False)
test_df = df[df["_rank"] == 1].copy()
val_df = df[df["_rank"] == 2].copy()
train_df = df[df["_rank"] > 2].copy()
df = df.drop(columns=["_rank"])
 
val_gt = val_df.groupby("user_id")["item_id"].apply(set).to_dict()
test_gt = test_df.groupby("user_id")["item_id"].apply(set).to_dict()
 
seen_items_val = train_df.groupby("user_id")["item_idx"].apply(set).to_dict()
seen_items_test = pd.concat([train_df, val_df]).groupby("user_id")["item_idx"].apply(set).to_dict()
 
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
 
assert "sentiment_score" in df.columns, "sentiment_score column not found!"
print(f"sentiment_score: found (mean={df['sentiment_score'].mean():.3f})")
 

Device: cuda
Rows: 563,929 | Users: 32,709 | Items: 2,000
Train: 498,511 | Val: 32,709 | Test: 32,709
sentiment_score: found (mean=0.620)


In [2]:
# SBERT item embeddings, item sentiment

from sentence_transformers import SentenceTransformer

# SBERT embeddings
sbert_path = OUT_DIR / "item_sbert_embeddings_mean5.npy"

if sbert_path.exists():
    print("Loading cached SBERT embeddings...")
    item_sbert = np.load(sbert_path)
else:
    print("Computing SBERT item embeddings (mean of last 5 train reviews per item)...")
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

    # last 5 train reviews per item
    item_reviews = (
        train_df.sort_values(["item_idx", "timestamp"], ascending=[True, False])
        .groupby("item_idx")["text_combined"]
        .apply(lambda texts: [t for t in texts.fillna("").astype(str).head(5) if t.strip()])
        .to_dict()
    )

    texts_to_encode = []
    item_ptrs = []
    for item_idx in range(n_items):
        reviews = item_reviews.get(item_idx, [])
        start = len(texts_to_encode)
        texts_to_encode.extend(reviews)
        end = len(texts_to_encode)
        item_ptrs.append((start, end))

    print(f"Total review texts to encode: {len(texts_to_encode):,}")

    if len(texts_to_encode) > 0:
        all_review_embs = sbert_model.encode(
            texts_to_encode,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=False,
        )
        all_review_embs = np.asarray(all_review_embs, dtype=np.float32)
    else:
        all_review_embs = np.zeros((0, 384), dtype=np.float32)

    item_sbert = np.zeros((n_items, 384), dtype=np.float32)
    for item_idx, (s, e) in enumerate(item_ptrs):
        if e > s:
            item_sbert[item_idx] = all_review_embs[s:e].mean(axis=0)

    norms = np.linalg.norm(item_sbert, axis=1, keepdims=True) + 1e-8
    item_sbert = item_sbert / norms

    np.save(sbert_path, item_sbert)
    del sbert_model
    torch.cuda.empty_cache()
    print(f"Saved: {sbert_path}")

item_sbert_tensor = torch.tensor(item_sbert, dtype=torch.float32).to(DEVICE)
print(f"SBERT embeddings: {item_sbert_tensor.shape}")

# item-level sentiment
item_sentiment = train_df.groupby("item_idx")["sentiment_score"].mean()
item_sentiment_arr = np.zeros(n_items, dtype=np.float32)
for idx, val in item_sentiment.items():
    item_sentiment_arr[idx] = float(val)

# z-score normalization
sent_mean = item_sentiment_arr.mean()
sent_std = item_sentiment_arr.std() + 1e-8
item_sentiment_arr = (item_sentiment_arr - sent_mean) / sent_std

item_sentiment_tensor = torch.tensor(item_sentiment_arr, dtype=torch.float32).to(DEVICE)
print(
    f"Item sentiment: shape={item_sentiment_tensor.shape}, "
    f"mean={item_sentiment_arr.mean():.3f}, std={item_sentiment_arr.std():.3f}"
)


Computing SBERT item embeddings (mean of last 5 train reviews per item)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total review texts to encode: 10,000


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Saved: /kaggle/working/item_sbert_embeddings_mean5.npy
SBERT embeddings: torch.Size([2000, 384])
Item sentiment: shape=torch.Size([2000]), mean=0.000, std=1.000


In [3]:
# metrics and model
 
def ndcg_at_k(recommended, relevant, k):
    if not relevant: return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0
 
def recall_at_k(recommended, relevant, k):
    if not relevant: return 0.0
    return len(set(recommended[:k]) & relevant) / len(relevant)
 
def precision_at_k(recommended, relevant, k):
    return len(set(recommended[:k]) & relevant) / k
 
def hr_at_k(recommended, relevant, k):
    return 1.0 if set(recommended[:k]) & relevant else 0.0
 
def evaluate_recommendations(recs, ground_truth, k=10):
    ndcgs, recalls, precisions, hrs = [], [], [], []
    for uid, rec_list in recs.items():
        gt = ground_truth.get(uid, set())
        if not gt: continue
        ndcgs.append(ndcg_at_k(rec_list, gt, k))
        recalls.append(recall_at_k(rec_list, gt, k))
        precisions.append(precision_at_k(rec_list, gt, k))
        hrs.append(hr_at_k(rec_list, gt, k))
    return {
        f"NDCG@{k}": np.mean(ndcgs), f"Recall@{k}": np.mean(recalls),
        f"Precision@{k}": np.mean(precisions), f"HR@{k}": np.mean(hrs),
        "n_users_evaluated": len(ndcgs),
    }
 
class BPRDataset(TorchDataset):
    def __init__(self, df, n_items):
        df = df[(df["user_idx"] >= 0) & (df["item_idx"] >= 0)].copy()
        self.users = df["user_idx"].values
        self.pos_items = df["item_idx"].values
        self.n_items = n_items
        self.user_items = df.groupby("user_idx")["item_idx"].apply(set).to_dict()
    def __len__(self): return len(self.users)
    def __getitem__(self, idx):
        user, pos = self.users[idx], self.pos_items[idx]
        seen = self.user_items.get(user, set())
        neg = random.randint(0, self.n_items - 1)
        while neg in seen: neg = random.randint(0, self.n_items - 1)
        return torch.tensor(user, dtype=torch.long), torch.tensor(pos, dtype=torch.long), torch.tensor(neg, dtype=torch.long)
 
# NCF + SBERT item embeddings
class NeuMF_SBERT(nn.Module):
    def __init__(self, n_users, n_items, emb_dim_gmf=32, emb_dim_mlp=32,
                 mlp_layers=None, dropout=0.1, sbert_dim=384):
        super().__init__()
        if mlp_layers is None:
            mlp_layers = [64, 32, 16]

        self.gmf_user = nn.Embedding(n_users, emb_dim_gmf)
        self.gmf_item = nn.Embedding(n_items, emb_dim_gmf)

        self.mlp_user = nn.Embedding(n_users, emb_dim_mlp)
        self.mlp_item = nn.Embedding(n_items, emb_dim_mlp)

        self.text_proj = nn.Linear(sbert_dim, emb_dim_mlp, bias=False)

        mlp_input = emb_dim_mlp * 2
        layers = []
        for out_dim in mlp_layers:
            layers += [nn.Linear(mlp_input, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            mlp_input = out_dim
        self.mlp = nn.Sequential(*layers)

        self.output_layer = nn.Linear(emb_dim_gmf + mlp_layers[-1], 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item]:
            nn.init.normal_(emb.weight, std=0.01)
        nn.init.xavier_uniform_(self.text_proj.weight)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m is not self.text_proj:
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, user_idx, item_idx, item_sbert):
        # GMF
        gmf_out = self.gmf_user(user_idx) * self.gmf_item(item_idx)

        # MLP
        item_cf = self.mlp_item(item_idx)
        item_txt = self.text_proj(item_sbert)
        item_fused = item_cf + item_txt

        mlp_in = torch.cat([self.mlp_user(user_idx), item_fused], dim=-1)
        mlp_out = self.mlp(mlp_in)

        combined = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.output_layer(combined).squeeze(-1)
 
# sentiment-CF
class NeuMF_Sentiment(nn.Module):
    def __init__(self, n_users, n_items, emb_dim_gmf=32, emb_dim_mlp=32,
                 mlp_layers=None, dropout=0.2):
        super().__init__()
        if mlp_layers is None: mlp_layers = [64, 32, 16]
 
        self.gmf_user = nn.Embedding(n_users, emb_dim_gmf)
        self.gmf_item = nn.Embedding(n_items, emb_dim_gmf)
        self.mlp_user = nn.Embedding(n_users, emb_dim_mlp)
        self.mlp_item = nn.Embedding(n_items, emb_dim_mlp)
 
        mlp_input = emb_dim_mlp * 2
        layers = []
        for out_dim in mlp_layers:
            layers += [nn.Linear(mlp_input, out_dim), nn.ReLU(), nn.Dropout(dropout)]
            mlp_input = out_dim
        self.mlp = nn.Sequential(*layers)
 
        self.output_layer = nn.Linear(emb_dim_gmf + mlp_layers[-1] + 1, 1)
        self._init_weights()
 
    def _init_weights(self):
        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item]:
            nn.init.normal_(emb.weight, std=0.01)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
 
    def forward(self, user_idx, item_idx, item_sentiment):
        gmf_out = self.gmf_user(user_idx) * self.gmf_item(item_idx)
        mlp_in = torch.cat([self.mlp_user(user_idx), self.mlp_item(item_idx)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        sent = item_sentiment.unsqueeze(-1)
        combined = torch.cat([gmf_out, mlp_out, sent], dim=-1)
        return self.output_layer(combined).squeeze(-1)
 

In [4]:
# config

EMB_DIM_GMF = 32
EMB_DIM_MLP = 32
MLP_LAYERS = [64, 32, 16]
DROPOUT = 0.2
LR = 1e-3
WEIGHT_DECAY = 1e-5
N_EPOCHS = 50
BATCH_SIZE = 2048
PATIENCE = 4
EVAL_EVERY = 2
K = 10

dataset = BPRDataset(train_df, n_items)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
all_item_idxs = np.arange(n_items)

VAL_USERS_FIXED = list(val_gt.keys())
if len(VAL_USERS_FIXED) > 3000:
    random.seed(SEED)
    VAL_USERS_FIXED = random.sample(VAL_USERS_FIXED, 3000)

print(f"Fixed quick-val subset: {len(VAL_USERS_FIXED):,} users")

def train_and_evaluate(model, model_name, score_fn):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_ndcg = -1.0
    no_improve = 0
    best_state = None
    history = []

    print(f"\nTraining {model_name}...")
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}\n")

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for users_b, pos_b, neg_b in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            users_b = users_b.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            neg_b = neg_b.to(DEVICE)

            pos_score = score_fn(model, users_b, pos_b)
            neg_score = score_fn(model, users_b, neg_b)
            loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(users_b)

        avg_loss = total_loss / len(dataset)

        if epoch % EVAL_EVERY == 0 or epoch == 1:
            model.eval()
            users = VAL_USERS_FIXED
            scores_list = []

            with torch.no_grad():
                all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
                for uid in users:
                    uidx = user2idx.get(uid, -1)
                    if uidx < 0:
                        continue

                    user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
                    sc = score_fn(model, user_t, all_items_t).cpu().numpy()

                    for sidx in seen_items_val.get(uid, set()):
                        sc[sidx] = -1e9

                    top = np.argsort(sc)[::-1][:K]
                    rec = [idx2item[i] for i in top]
                    scores_list.append(ndcg_at_k(rec, val_gt[uid], K))

            val_ndcg = float(np.mean(scores_list)) if scores_list else 0.0
            print(f"  Epoch {epoch:>3}/{N_EPOCHS}  BPR: {avg_loss:.4f}  val_NDCG@{K}: {val_ndcg:.4f}")
            history.append({"epoch": epoch, "loss": avg_loss, "val_ndcg": val_ndcg})

            if val_ndcg > best_ndcg + 1e-5:
                best_ndcg = val_ndcg
                no_improve = 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    print(f"\n  Early stopping at epoch {epoch} (best: {best_ndcg:.4f})")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(DEVICE)

    print(f"  Best val NDCG@{K}: {best_ndcg:.4f}")

    @torch.no_grad()
    def generate_recs(target_users, seen_dict):
        model.eval()
        all_items_t = torch.tensor(all_item_idxs, dtype=torch.long, device=DEVICE)
        recs = {}

        for uid in tqdm(target_users, desc="Recs", leave=False):
            uidx = user2idx.get(uid, -1)
            if uidx < 0:
                continue

            user_t = torch.full((n_items,), uidx, dtype=torch.long, device=DEVICE)
            sc = score_fn(model, user_t, all_items_t).cpu().numpy()

            for sidx in seen_dict.get(uid, set()):
                sc[sidx] = -1e9

            top = np.argsort(sc)[::-1][:K]
            recs[uid] = [idx2item[i] for i in top]

        return recs

    print(f"\nEvaluating {model_name} on VAL...")
    val_recs = generate_recs(val_df["user_id"].unique().tolist(), seen_items_val)
    val_metrics = evaluate_recommendations(val_recs, val_gt, k=K)

    print(f"Evaluating {model_name} on TEST...")
    test_recs = generate_recs(test_df["user_id"].unique().tolist(), seen_items_test)
    test_metrics = evaluate_recommendations(test_recs, test_gt, k=K)

    print(f"\n{'='*40}")
    print(f"{model_name} — VAL")
    print(f"{'='*40}")
    for key, val in val_metrics.items():
        print(f"  {key:<20} {val:.4f}" if isinstance(val, float) else f"  {key:<20} {val}")

    print(f"\n{model_name} — TEST")
    print(f"{'='*40}")
    for key, val in test_metrics.items():
        print(f"  {key:<20} {val:.4f}" if isinstance(val, float) else f"  {key:<20} {val}")

    fname = model_name.lower().replace("+", "_").replace("-", "_").replace(" ", "_")
    results = {"model": model_name, "val": val_metrics, "test": test_metrics}

    with open(OUT_DIR / f"{fname}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    torch.save(model.state_dict(), OUT_DIR / f"{fname}_model.pt")
    pd.DataFrame(history).to_csv(OUT_DIR / f"{fname}_history.csv", index=False)

    print(f"Saved: {fname}_results.json, {fname}_model.pt, {fname}_history.csv")
    return val_metrics, test_metrics, history


Fixed quick-val subset: 3,000 users


In [5]:
# train cf+bert

model_cfbert = NeuMF_SBERT(
    n_users=n_users,
    n_items=n_items,
    emb_dim_gmf=32,
    emb_dim_mlp=32,
    mlp_layers=[64, 32, 16],
    dropout=0.1,
    sbert_dim=384,
).to(DEVICE)

def score_cfbert(model, user_idx, item_idx):
    sbert = item_sbert_tensor[item_idx]
    return model(user_idx, item_idx, sbert)

cfbert_val, cfbert_test, cfbert_hist = train_and_evaluate(
    model_cfbert, "CF+BERT", score_cfbert
)


Training CF+BERT...
Params: 2,240,481



  Epoch   1/50  BPR: 0.6155  val_NDCG@10: 0.0042


  Epoch   2/50  BPR: 0.4555  val_NDCG@10: 0.0072


  Epoch   4/50  BPR: 0.3352  val_NDCG@10: 0.0120


  Epoch   6/50  BPR: 0.2791  val_NDCG@10: 0.0190


...

  Epoch  32/50  BPR: 0.1274  val_NDCG@10: 0.0338


  Epoch  34/50  BPR: 0.1258  val_NDCG@10: 0.0357


  Epoch  36/50  BPR: 0.1230  val_NDCG@10: 0.0359

  Early stopping at epoch 36 (best: 0.0377)
  Best val NDCG@10: 0.0377

Evaluating CF+BERT on VAL...


Evaluating CF+BERT on TEST...



CF+BERT — VAL
  NDCG@10              0.0390
  Recall@10            0.0778
  Precision@10         0.0078
  HR@10                0.0778
  n_users_evaluated    32709

CF+BERT — TEST
  NDCG@10              0.0263
  Recall@10            0.0537
  Precision@10         0.0054
  HR@10                0.0537
  n_users_evaluated    32709
Saved: cf_bert_results.json, cf_bert_model.pt, cf_bert_history.csv


In [6]:
# sentiment-cf

model_sentcf = NeuMF_Sentiment(
    n_users=n_users, n_items=n_items,
    emb_dim_gmf=EMB_DIM_GMF, emb_dim_mlp=EMB_DIM_MLP,
    mlp_layers=MLP_LAYERS, dropout=DROPOUT,
).to(DEVICE)

def score_sentcf(model, user_idx, item_idx):
    sent = item_sentiment_tensor[item_idx]
    return model(user_idx, item_idx, sent)

sentcf_val, sentcf_test, sentcf_hist = train_and_evaluate(
    model_sentcf, "Sentiment-CF", score_sentcf
)


Training Sentiment-CF...
Params: 2,228,194



  Epoch   1/50  BPR: 0.6502  val_NDCG@10: 0.0033


  Epoch   2/50  BPR: 0.5222  val_NDCG@10: 0.0048


  Epoch   4/50  BPR: 0.4032  val_NDCG@10: 0.0097


  Epoch   6/50  BPR: 0.3256  val_NDCG@10: 0.0137


...

  Epoch  40/50  BPR: 0.1479  val_NDCG@10: 0.0346


  Epoch  42/50  BPR: 0.1472  val_NDCG@10: 0.0346


  Epoch  44/50  BPR: 0.1439  val_NDCG@10: 0.0342

  Early stopping at epoch 44 (best: 0.0379)
  Best val NDCG@10: 0.0379

Evaluating Sentiment-CF on VAL...


Evaluating Sentiment-CF on TEST...



Sentiment-CF — VAL
  NDCG@10              0.0374
  Recall@10            0.0749
  Precision@10         0.0075
  HR@10                0.0749
  n_users_evaluated    32709

Sentiment-CF — TEST
  NDCG@10              0.0244
  Recall@10            0.0503
  Precision@10         0.0050
  HR@10                0.0503
  n_users_evaluated    32709
Saved: sentiment_cf_results.json, sentiment_cf_model.pt, sentiment_cf_history.csv
